# Семинар 9: функции и сортировка

### Функции

Функцией будем считать некоторый обособленный кусок кода, который можно вызвать из любой другой части кода.

In [ ]:
def has_negative_number(list_numbers):
    for num in list_numbers:
        if num < 0:
            # ключевое слово return сразу говорит питону выйти из функции и вернуть значение
            return True

    return False  # вопрос: что вернет функция, если я забуду здесь return False написать?

In [ ]:
print(has_negative_number([1, 5, 2, 3]))

In [ ]:
# тайпинги все еще сохраняют динамическую типизацию,
# но позволяют улучшить читаемость кода
# чекеры вроде mypy умеют анализировать код и проверять их корректность

from typing import Iterable

def has_negative_number(numbers: Iterable[int]) -> bool:
    for num in numbers:
        if num < 0:
            return True

    return False

In [ ]:
with open("numbers.txt", "r") as fin:
    while (line := fin.readline()):
        numbers = list(map(int, line.split()))
        print(f"{numbers} -> {has_negative_number(numbers)}")

Функцию можно сделать чуть более удобной для использования не только списком:

In [65]:
# все аргументы через * типизируются без List
# https://peps.python.org/pep-0484/#arbitrary-argument-lists-and-default-argument-values

def has_negative_number(*numbers: int) -> bool:
    for num in numbers:
        if num < 0:
            return True

    return False

print(has_negative_number(1, 2, 3, 4, -1))  # теперь можно не передавать список

True


In [ ]:
print(1, 2, 3, 4, 5)

Помимо `*args` есть еще и `**kwargs`:

In [ ]:
# на самом деле, в качестве тайпинга для values подошло бы
# какое-нибудь самописное comparable, но мы тут не будем с этим заморачиваться
# https://stackoverflow.com/questions/37669222/how-can-i-hint-that-a-type-is-comparable-with-typing


def max_value(*values: int | str, **params: bool) -> int | str | None:
    return_idx = params.get("return_idx", False)  # достаем позиционный аргумент

    if not values:  # если список пустой, вернем None
        return None

    max_value = values[0]
    max_value_idx = 0
    for i, value in enumerate(values):
        if value > max_value:
            max_value = value
            max_value_idx = i

    if return_idx:
        return max_value_idx
    return max_value

In [ ]:
print("max value:", max_value(6, -1, 2, 9, 1))  # сам максимум
print("max value index:", max_value(6, -1, 2, 9, 1, return_idx=True)) # индекс максимума

max value: 9
max value index: 3


In [ ]:
max_value(6, "a")

In [70]:
# на самом деле, в качестве тайпинга для values подошло бы
# какое-нибудь самописное comparable, но мы тут не будем с этим заморачиваться
# https://stackoverflow.com/questions/37669222/how-can-i-hint-that-a-type-is-comparable-with-typing

from typing import TypeVar

# T = TypeVar("T", bound=int | str)

# type T = int | str

# def max_value(*values: T, return_idx: bool = False) -> T | None:
def max_value[T: int | str](*values: T, return_idx: bool = False) -> T | None:
    if not values:  # если список пустой, вернем None
        return None

    max_value = values[0]
    max_value_idx = 0
    for i, value in enumerate(values):
        if value > max_value:
            max_value = value
            max_value_idx = i

    if return_idx:
        return max_value_idx
    return max_value

In [71]:
print("max value:", max_value(6, -1, 2, 9, 1))  # сам максимум
print("max value index:", max_value(6, -1, 2, 9, 1, return_idx=True)) # индекс максимума

max value: 9
max value index: 3


In [73]:
def my_max(a, b):
    if a > b:
        return a
    return b

In [ ]:
my_max(-1, 1)

1

### Рестрикшены

В аргументы функции можно написать пустую `*`. Тогда будет считаться, что все после этой звездочки -- это именованные аргументы и больше позиционные использовать нельзя. Ранее мы могли вызвать функцию вот так:

In [80]:
def my_max(a, b, print_hello=True, **kwargs):
    if print_hello:
        print(a, b, **kwargs)

    if a > b:
        return a
    return b


my_max(1, 2, print_hello=True)

1 2


2

А если добавить, то уже нельзя:

In [81]:
def my_max(a, b, *, print_hello=True, **kwargs):
    if print_hello:
        print(a, b, **kwargs)

    if a > b:
        return a
    return b

In [82]:
my_max(1, 2, print_hello=True)

1 2


2

In [83]:
my_max(1, 2, True, sep='.', end='!')  # ошибка

TypeError: my_max() takes 2 positional arguments but 3 were given

In [84]:
my_max(1, 2, print_hello=True, sep='.', end='!')  # все ОК

1.2!

2

In [ ]:
my_max(a=1, b=2, print_hello=True, sep='.', end='!')  # и так тоже работать будет

Можно еще лучше: давайте запретим первым двум аргументам быть именованными в принципе. Это делается через `/`:

In [87]:
def my_max(a, b, /, c, *, print_hello=True, **kwargs):
    if print_hello:
        print(a, b, c, **kwargs)

    if a > b:
        return a
    return b

In [88]:
my_max(a=1, b=2, c=3, print_hello=True, sep='.', end='!')  # ошибка

TypeError: my_max() missing 2 required positional arguments: 'a' and 'b'

In [89]:
my_max(1, 2, 3, print_hello=True, sep='.', end='!')  # все работает

1.2.3!

2

In [ ]:
my_max(1, 2, c=3, print_hello=True, sep='.', end='!')  # и так тоже работает

In [91]:
max(arg1=1, arg2=2)

TypeError: max expected at least 1 argument, got 0

### Области видимости

По умолчанию, все переменные в питоне, которые объявлены внутри функции, локальные. Вот парочка игрушечных примеров

In [92]:
def f(x):
    square_x = x ** 2
    print(square_x)

f(10)

print(square_x)  # ну конечно, вам скорее всего даже редактор подчеркнет, что так низя

100


NameError: name 'square_x' is not defined

In [93]:
value = 10

def f():
    print(value ** 2)  # но читать переменные из глобальной области можно

f()

100


In [94]:
value = 10

def f():
    value += 1  # но примитивы менять низя (вы зачем вообще это делать хотите???)

f()

UnboundLocalError: cannot access local variable 'value' where it is not associated with a value

In [ ]:
value = 10

def f():
    global value  # глобал -- это очень и очень плохо
    # иногда полезно делать global для каких-то совсем важных штук, но это редкость
    value += 1  # но если очень хочется, то все-таки можно

f()

print(value)

In [ ]:
credentials = None

def create_user():
    global credentials  # изначально None и поэтому просто так не обратимся

    credentials = {
        "login": "destroyer_2007",
        "password": "zavtra_v_shkolu((((",
    }

print(credentials)

create_user()

print(credentials)


In [96]:
a = [1, 2, 3]  # a -----> [1, 2, 3, 4]

def f():
    a.append(4)  # но вы же не думали, что все так просто... подумайте, почему так происходит

f()

print(a)

[1, 2, 3, 4]


In [ ]:
def f(a: list):  # равно так же как и поменяет список передача списка в функцию (ссылки же)
    a.append(4)

a = [1, 2, 3]

f(a)

print(a)

In [99]:
x = 6

def f():
    x = 1

    def g():
        nonlocal x
        x += 1
        print(x)

    return g

g = f()
g()
g()
g()

2
3
4


In [100]:
def counter(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        result = func(*args, **kwargs)
        count += 1
        print("function", func.__name__, "was called", count, "times")
        return result
    return wrapper


@counter
def f(x):
    return x + 1

print(f(1))
print(f(2))
print(f(3))

function f was called 1 times
2
function f was called 2 times
3
function f was called 3 times
4


### Лямбды

Анонимные однострочные функции

In [102]:
# вообще, присваивать лямбды переменным плохой тон

is_negative = lambda x: x < 0

is_negative(1)

False

In [103]:
next_value = lambda n: n + 1

print(next_value(1))

2


In [104]:
(lambda *args: sum(args) / len(args))(5, 8, 7, 9)

7.25

In [105]:
(lambda x, y: x + y)(5, 8)

13

In [ ]:
def square(x: int) -> int:
    return x ** 2


values = [1, 2, 3, 4, 5, 6]

print(list(map(square, values)))

In [ ]:
values = [1, 2, 3, 4, 5, 6]

print(list(map(lambda x: x ** 2, values)))

### Сортировка

Мы уже говорили про сортировку списков, но не говорили, что ей можно задать кастомный ключ сравнения через функции

In [106]:
values = ["abcd", "aab", "bda", "0xabadbabe", "0xdeadbeef"]

sorted(values)  # все то же самое далее можно делать и с values.sort()

['0xabadbabe', '0xdeadbeef', 'aab', 'abcd', 'bda']

In [ ]:
values = ["abcd", "aab", "bda", "0xabadbabe", "0xdeadbeef"]

sorted(values, key=lambda s: len(s))  # все то же самое далее можно делать и с values.sort()

In [ ]:
values = ["abcd", "aab", "bda", "0xabadbabe", "0xdeadbeef"]

sorted(values, key=len)  # все то же самое далее можно делать и с values.sort()

А как нам сравнивать по нескольким значениям сразу? Например, есть числа, хотим отсортировать их по возрастанию длин, но при равенстве -- по убыванию самих чисел

In [112]:
values = [7876510, 678, 456789, 789, 123456]

sorted(values, key=lambda num: (len(str(num)), -num))  # кортежи сравниваются поэлементно, поэтому это так и работает

[789, 678, 456789, 123456, 7876510]

In [110]:
(1, 4) < (-1, 4, 6)

False

In [111]:
print([7876510, 678, 456789, 789, 123456])
print(list(map(lambda num: (len(str(num)), -num), [7876510, 678, 456789, 789, 123456])))

[7876510, 678, 456789, 789, 123456]
[(7, -7876510), (3, -678), (6, -456789), (3, -789), (6, -123456)]


In [ ]:
from math import sqrt

type Point = tuple[int, int]


def perimiter(a: Point, b: Point, c: Point, /) -> float:
    perimeter_sum = 0
    for (first, second) in ((a, b), (b, c), (a, c)):
        perimeter_sum += sqrt((first[0] - second[0]) ** 2 + (first[1] - second[1]) ** 2)
    return perimeter_sum


x1, y1 = int(input()), int(input())
x2, y2 = int(input()), int(input())
x3, y3 = int(input()), int(input())
print(perimiter((x1, y1), (x2, y2), (x3, y3)))

3.414213562373095